# NUS DSA3361 — Inferential Data Analytics
## Kaggle Case Study: Telecom Customer Churn
### **Inference first; prediction second**

This notebook uses the well-known **Kaggle Telco Customer Churn** dataset (`blastchar/telco-customer-churn`) as an applied DSA3361-style case study.

The dataset contains **7,043 customers and 21 original columns**. Each row represents a customer. The outcome `Churn` indicates whether the customer left within the last month.

The raw variables cover:

- demographics;
- tenure;
- subscribed telecom/internet services;
- contract type;
- payment method;
- paperless billing;
- monthly and total charges;
- churn status.

---

# The central distinction

We will intentionally ask **two different questions** of the same dataset.

## Inferential question

> Which observed customer characteristics are associated with churn, **by how much**, and **with how much uncertainty**, after adjusting for other measured characteristics?

Typical outputs:

\[
\widehat{\Delta},\quad
RR,\quad OR,\quad
SE,\quad CI,\quad
p\text{-value}
\]

and model diagnostics.

## Predictive question

> How accurately can we estimate churn risk for a new customer drawn from a comparable population?

Typical outputs:

\[
\hat p_i=P(Y_i=1\mid X_i),
\]

cross-validation scores, ROC-AUC, PR-AUC, Brier score, calibration and decision thresholds.

---

# Why the distinction matters

A variable can be:

- statistically associated with churn but contribute little predictive lift;
- highly predictive without having a simple scientific interpretation;
- associated with churn without being a causal driver.

Therefore:

\[
\boxed{
\text{statistical significance}
\neq
\text{predictive usefulness}
\neq
\text{causal importance}
}
\]

This notebook emphasizes the **first** of those: inferential analytics.

## Learning objectives

After completing the notebook, you should be able to:

1. formulate an **estimand** before choosing an algorithm;
2. distinguish descriptive statistics from inferential claims;
3. estimate churn prevalence with confidence intervals;
4. compare groups using:
   - risk difference;
   - risk ratio;
   - odds ratio;
   - chi-square tests;
   - Welch t-tests;
   - standardized effect sizes;
5. use **bootstrap resampling** to quantify estimator uncertainty;
6. use **permutation tests** to construct a null distribution;
7. deal with **multiple hypothesis testing** using Benjamini–Hochberg FDR control;
8. identify confounding and explain why unadjusted and adjusted effects differ;
9. fit an adjusted logistic/GLM model with robust standard errors;
10. interpret coefficients as odds ratios with confidence intervals;
11. compute model-standardized adjusted churn probabilities;
12. test interaction/effect modification;
13. diagnose:
    - multicollinearity;
    - nonlinear functional form;
    - influential observations;
    - calibration;
14. explain why this observational dataset does **not** establish causality;
15. build a leakage-safe predictive pipeline;
16. compare inferential conclusions with predictive performance.

---

## Dataset references

- Kaggle: `https://www.kaggle.com/datasets/blastchar/telco-customer-churn`
- Original filename: `WA_Fn-UseC_-Telco-Customer-Churn.csv`

For coursework or reproducibility, download the Kaggle CSV and place it beside this notebook or in a `data/` directory.

# Analytical roadmap

```text
                 BUSINESS / SCIENTIFIC QUESTION
                            |
                            v
                     DATA QUALITY AUDIT
                            |
              +-------------+-------------+
              |                           |
              v                           v
        INFERENTIAL                    PREDICTIVE
         ANALYTICS                     ANALYTICS
              |                           |
       define estimands             define target/loss
              |                           |
       effect sizes + CI            train/test split
              |                           |
    bootstrap / permutation         preprocessing pipeline
              |                           |
    adjusted regression             cross-validation
              |                           |
  interaction + diagnostics         held-out metrics
              |                           |
  uncertainty + limitations         calibration/threshold
              +-------------+-------------+
                            |
                            v
                 DECISION + COMMUNICATION
```

The notebook deliberately completes most of the **left-hand branch** before moving to prediction.

# 0. Environment

Main libraries:

- `pandas`, `numpy`
- `scipy`
- `statsmodels` — classical inference and GLMs
- `scikit-learn` — predictive modelling
- `bokeh` — interactive visualisation

If required, uncomment the installation cell.

In [1]:
# %pip install -q pandas numpy scipy statsmodels scikit-learn bokeh kagglehub

In [2]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Optional
import warnings

import numpy as np
from numpy.linalg import LinAlgError
import pandas as pd

from scipy import stats

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
)
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.calibration import calibration_curve

from bokeh.io import output_notebook, show
from bokeh.layouts import gridplot
from bokeh.models import ColumnDataSource, HoverTool, Span, Whisker
from bokeh.plotting import figure, show
from bokeh.palettes import Category10
import patsy


warnings.filterwarnings("ignore")
output_notebook()

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)


@dataclass(frozen=True)
class Config:
    test_size: float = 0.20
    bootstrap_reps: int = 5_000
    permutation_reps: int = 10_000
    adjusted_bootstrap_reps: int = 500
    cv_splits: int = 5
    cv_repeats: int = 3


CFG = Config()
CFG

Loading BokehJS ...

Config(test_size=0.2, bootstrap_reps=5000, permutation_reps=10000, adjusted_bootstrap_reps=500, cv_splits=5, cv_repeats=3)

# 1. Load the Kaggle dataset

The loader checks several common locations:

1. current directory;
2. `data/`;
3. Kaggle Notebook mounted path;
4. optional `kagglehub` download.

The target file is:

`WA_Fn-UseC_-Telco-Customer-Churn.csv`

The notebook intentionally does **not** silently fabricate data if loading fails.

In [3]:
class TelcoDatasetLoader:
    FILE_NAME = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
    KAGGLE_HANDLE = "blastchar/telco-customer-churn"

    def __init__(self) -> None:
        self.candidates = [
            Path(self.FILE_NAME),
            Path("data") / self.FILE_NAME,
            Path("/kaggle/input/telco-customer-churn") / self.FILE_NAME,
        ]

    def load(self) -> pd.DataFrame:
        for path in self.candidates:
            if path.exists():
                print(f"Loading dataset from: {path}")
                return pd.read_csv(path)

        # Optional public Kaggle download path.
        try:
            import kagglehub
            root = Path(kagglehub.dataset_download(self.KAGGLE_HANDLE))
            candidate = root / self.FILE_NAME
            if candidate.exists():
                print(f"Downloaded through kagglehub: {candidate}")
                return pd.read_csv(candidate)
        except Exception as exc:
            print("kagglehub download not available:", type(exc).__name__)

        raise FileNotFoundError(
            f"Could not locate {self.FILE_NAME}. "
            "Download it from Kaggle dataset "
            "'blastchar/telco-customer-churn' and place it beside this notebook "
            "or inside ./data/."
        )


raw = TelcoDatasetLoader().load()
print("Raw shape:", raw.shape)
display(raw.head())

Downloaded through kagglehub: /home/anirban/.cache/kagglehub/datasets/blastchar/telco-customer-churn/versions/1/WA_Fn-UseC_-Telco-Customer-Churn.csv
Raw shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# 2. Data quality audit

Inference is only as defensible as the data-generating assumptions.

Before hypothesis tests or regression, inspect:

- column types;
- duplicate rows;
- customer identifier uniqueness;
- missing values;
- impossible values;
- target prevalence;
- hidden missingness represented as blank strings.

A well-known issue in this dataset is that `TotalCharges` may be read as text because some rows contain blank values.

In [4]:
COLUMN_MAP = {
    "customerID": "customer_id",
    "gender": "gender",
    "SeniorCitizen": "senior_citizen",
    "Partner": "partner",
    "Dependents": "dependents",
    "tenure": "tenure",
    "PhoneService": "phone_service",
    "MultipleLines": "multiple_lines",
    "InternetService": "internet_service",
    "OnlineSecurity": "online_security",
    "OnlineBackup": "online_backup",
    "DeviceProtection": "device_protection",
    "TechSupport": "tech_support",
    "StreamingTV": "streaming_tv",
    "StreamingMovies": "streaming_movies",
    "Contract": "contract",
    "PaperlessBilling": "paperless_billing",
    "PaymentMethod": "payment_method",
    "MonthlyCharges": "monthly_charges",
    "TotalCharges": "total_charges",
    "Churn": "churn",
}

df = raw.rename(columns=COLUMN_MAP).copy()

missing_required = sorted(set(COLUMN_MAP.values()) - set(df.columns))
if missing_required:
    raise ValueError(f"Missing expected columns: {missing_required}")

df["total_charges_raw"] = df["total_charges"]
df["total_charges"] = pd.to_numeric(df["total_charges"], errors="coerce")
df["churn_num"] = df["churn"].map({"No": 0, "Yes": 1}).astype("int64")

print("Shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["customer_id"].duplicated().sum())
print("Missing TotalCharges after numeric coercion:", df["total_charges"].isna().sum())
print("Overall churn rate:", df["churn_num"].mean())

Shape: (7043, 23)
Duplicate rows: 0
Duplicate customer IDs: 0
Missing TotalCharges after numeric coercion: 11
Overall churn rate: 0.2653698707936959


In [5]:
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_n": df.isna().sum(),
    "missing_pct": (100 * df.isna().mean()).round(3),
    "n_unique": df.nunique(dropna=True),
}).sort_values(["missing_pct", "n_unique"], ascending=[False, True])

display(audit)

,dtype,missing_n,missing_pct,n_unique
total_charges,float64,11,0.156,6530
gender,object,0,0.000,2
senior_citizen,int64,0,0.000,2
partner,object,0,0.000,2
dependents,object,0,0.000,2
phone_service,object,0,0.000,2
paperless_billing,object,0,0.000,2
churn,object,0,0.000,2
churn_num,int64,0,0.000,2
multiple_lines,object,0,0.000,3


## Why missingness matters for inference

Suppose rows with missing `TotalCharges` are not random—for example, they systematically correspond to very new customers.

Then a complete-case analysis could estimate relationships for a selected subset rather than the full intended population.

We will:

- **not** use `TotalCharges` in the main inferential model;
- use transparent complete cases where needed;
- use pipeline imputation for predictive modelling.

These choices serve different goals.

For a formal missing-data study you would ask whether missingness is approximately:

- MCAR;
- MAR;
- MNAR;

and potentially use multiple imputation.

# 3. Analysis-friendly variables

We create interpretable transformed variables:

\[
TenureYears = \frac{tenure}{12}
\]

\[
MonthlyCharges10 = \frac{MonthlyCharges}{10}
\]

so regression coefficients correspond to:

- one **year** of tenure;
- a **$10** monthly-charge difference.

We also make clean binary indicators for selected Yes/No variables.

In [6]:
def prepare_analysis_frame(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()

    out["tenure_years"] = out["tenure"] / 12.0
    out["monthly_charges_10"] = out["monthly_charges"] / 10.0

    for col in ["partner", "dependents", "paperless_billing", "phone_service"]:
        out[f"{col}_yes"] = out[col].map({"No": 0, "Yes": 1})

    out["is_month_to_month"] = (out["contract"] == "Month-to-month").astype(int)
    out["is_two_year"] = (out["contract"] == "Two year").astype(int)

    out["tenure_band"] = pd.cut(
        out["tenure"],
        bins=[-0.1, 12, 24, 48, 60, np.inf],
        labels=["0–12", "13–24", "25–48", "49–60", "61+"],
    )

    return out


df = prepare_analysis_frame(df)
display(df.head())

,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,churn_num,tenure_years,monthly_charges_10,partner_yes,dependents_yes,paperless_billing_yes,phone_service_yes,is_month_to_month,is_two_year,tenure_band
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,0,0.083333,2.985,1,0,1,0,1,0,0–12
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,0,2.833333,5.695,0,0,0,1,0,0,25–48
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,1,0.166667,5.385,0,0,1,1,1,0,0–12
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,0,3.750000,4.230,0,0,0,0,0,0,25–48
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,1,0.166667,7.070,0,0,1,1,1,0,0–12


# 🧭 Part I — Inferential Analytics

## 🎯 Define the Target Population, Estimands, and Assumptions

Before touching a p‑value, define what you are trying to estimate.

> **Descriptive target population:** customers represented by the IBM/Kaggle telecom sample, or customers from a comparable operating environment.  
> ⚠️ Do **not** automatically generalize to every telecom company, country, or time period.

---

### 📈 Estimand 1 — Churn prevalence



$p = P(Y = 1)$




Estimate:




$\hat{p} = \frac{1}{n}\sum_i Y_i$


---

### 🔄 Estimand 2 — Unadjusted contract risk difference

For month‑to‑month versus two‑year contracts:



$RD = P(Y = 1 \mid M2M) - P(Y = 1 \mid TwoYear)$



---

### ⚖️ Estimand 3 — Unadjusted risk ratio



$RR = \frac{P(Y = 1 \mid M2M)}{P(Y = 1 \mid TwoYear)}$


---

### 🧮 Estimand 4 — Adjusted association

For logistic regression:



$\log\left(\frac{P(Y = 1 \mid X)}{1 - P(Y = 1 \mid X)}\right) = X\beta$



We use:


$OR_j = e^{\beta_j}$



to summarize adjusted conditional associations.

---

### 📊 Estimand 5 — Model‑standardized churn probability

For contract level \(c\):



$\hat{P}_c = \frac{1}{n}\sum_{i=1}^{n}\hat{P}(Y_i = 1 \mid Contract = c, X_{i,-contract})$



This is often easier for business stakeholders to interpret than an odds ratio.

---

### ⚠️ Critical Caution

These standardized quantities are still **model‑based associations**.  
Without stronger design and causal assumptions, we should **not** say:

> “Changing a customer’s contract *causes* churn to fall by X percentage points.”


# 5. Overall churn prevalence and its uncertainty

The sample churn proportion is:

\[
\hat p = \frac{x}{n}.
\]

Instead of reporting only \(\hat p\), construct a 95% **Wilson confidence interval**.

This represents uncertainty due to sampling variability under the binomial-style model.

In [7]:
n = len(df)
x = int(df["churn_num"].sum())
p_hat = x / n

ci_low, ci_high = proportion_confint(
    count=x,
    nobs=n,
    alpha=0.05,
    method="wilson",
)

print(f"Customers: {n:,}")
print(f"Churners: {x:,}")
print(f"Observed churn rate: {p_hat:.4f}")
print(f"95% Wilson CI: [{ci_low:.4f}, {ci_high:.4f}]")

Customers: 7,043
Churners: 1,869
Observed churn rate: 0.2654
95% Wilson CI: [0.2552, 0.2758]


### Interpretation template

A defensible statement is:

> The observed churn rate is \(\hat p\). Under the sampling assumptions used by the interval procedure, the 95% confidence interval is \([L,U]\).

Avoid:

> There is a 95% probability that the fixed population parameter lies inside this already-computed frequentist interval.

The repeated-sampling interpretation belongs to the **procedure**.

# 6. Descriptive churn rates with confidence intervals

Before formal testing, inspect the **magnitude** of group differences.

The function below calculates:

- group count;
- churn count;
- churn rate;
- Wilson CI.

We visualize contract, internet service and payment method.

In [8]:
def churn_rate_table(
    data: pd.DataFrame,
    group_col: str,
) -> pd.DataFrame:
    g = (
        data.groupby(group_col, dropna=False)["churn_num"]
        .agg(["count", "sum", "mean"])
        .reset_index()
        .rename(columns={"mean": "churn_rate"})
    )

    intervals = [
        proportion_confint(
            count=int(row["sum"]),
            nobs=int(row["count"]),
            alpha=0.05,
            method="wilson",
        )
        for _, row in g.iterrows()
    ]

    g["ci_low"] = [ci[0] for ci in intervals]
    g["ci_high"] = [ci[1] for ci in intervals]
    return g


contract_rates = churn_rate_table(df, "contract")
internet_rates = churn_rate_table(df, "internet_service")
payment_rates = churn_rate_table(df, "payment_method")

display(contract_rates)
display(internet_rates)
display(payment_rates)

,contract,count,sum,churn_rate,ci_low,ci_high
0,Month-to-month,3875,1655,0.427097,0.411602,0.442736
1,One year,1473,166,0.112695,0.097544,0.129862
2,Two year,1695,48,0.028319,0.021425,0.037345


,internet_service,count,sum,churn_rate,ci_low,ci_high
0,DSL,2421,459,0.189591,0.174474,0.205692
1,Fiber optic,3096,1297,0.418928,0.401659,0.436397
2,No,1526,113,0.074050,0.061954,0.088284


,payment_method,count,sum,churn_rate,ci_low,ci_high
0,Bank transfer (automatic),1544,258,0.167098,0.149321,0.186528
1,Credit card (automatic),1522,232,0.152431,0.135250,0.171362
2,Electronic check,2365,1071,0.452854,0.432885,0.472976
3,Mailed check,1612,308,0.191067,0.172618,0.210984


In [9]:
def rate_plot(
    table: pd.DataFrame,
    category: str,
    title: str,
    width: int = 650,
):
    t = table.copy()
    t[category] = t[category].astype(str)

    source = ColumnDataSource(t)

    p = figure(
        x_range=t[category].tolist(),
        height=390,
        width=width,
        title=title,
        toolbar_location=None,
        y_axis_label="Observed churn proportion",
    )

    p.vbar(
        x=category,
        top="churn_rate",
        width=0.65,
        source=source,
    )

    p.add_layout(
        Whisker(
            base=category,
            upper="ci_high",
            lower="ci_low",
            source=source,
        )
    )

    p.y_range.start = 0
    p.y_range.end = min(1.0, max(0.6, float(t["ci_high"].max()) + 0.1))

    p.add_tools(HoverTool(
        tooltips=[
            (category, f"@{category}"),
            ("n", "@count{0,0}"),
            ("churners", "@sum{0,0}"),
            ("rate", "@churn_rate{0.000}"),
            ("95% CI", "[@ci_low{0.000}, @ci_high{0.000}]"),
        ]
    ))

    return p


show(rate_plot(
    contract_rates,
    "contract",
    "Observed churn rate by contract type",
))

# 7. Effect sizes: risk difference, risk ratio and odds ratio

Let's compare:

\[
A=\text{Month-to-month contract}
\]

with

\[
B=\text{Two-year contract}.
\]

A hypothesis test alone does not answer **how large** the difference is.

We therefore estimate three effect measures.

---

## Risk difference

\[
RD=p_A-p_B
\]

Directly interpretable in **percentage points**.

---

## Risk ratio

\[
RR=\frac{p_A}{p_B}
\]

Interpretation: group A has \(RR\) times the observed churn risk of group B.

---

## Odds ratio

\[
OR=
\frac{p_A/(1-p_A)}
{p_B/(1-p_B)}.
\]

Odds ratios are convenient for logistic regression but are not the same as risk ratios.

In [10]:
def two_group_effect_table(
    data: pd.DataFrame,
    group_col: str,
    group_a: str,
    group_b: str,
) -> pd.DataFrame:
    subset = data[data[group_col].isin([group_a, group_b])]

    rows = []
    for group in [group_a, group_b]:
        y = subset.loc[subset[group_col] == group, "churn_num"]
        rows.append({
            "group": group,
            "n": len(y),
            "churners": int(y.sum()),
            "non_churners": int((1 - y).sum()),
            "risk": y.mean(),
        })

    return pd.DataFrame(rows)


contract_2x2 = two_group_effect_table(
    df,
    group_col="contract",
    group_a="Month-to-month",
    group_b="Two year",
)

display(contract_2x2)

a = contract_2x2.iloc[0]
b = contract_2x2.iloc[1]

risk_a = a["risk"]
risk_b = b["risk"]

risk_difference = risk_a - risk_b
risk_ratio = risk_a / risk_b

odds_a = a["churners"] / a["non_churners"]
odds_b = b["churners"] / b["non_churners"]
odds_ratio = odds_a / odds_b

print(f"Risk difference: {risk_difference:.4f}")
print(f"Risk ratio:      {risk_ratio:.4f}")
print(f"Odds ratio:      {odds_ratio:.4f}")

,group,n,churners,non_churners,risk
0,Month-to-month,3875,1655,2220,0.427097
1,Two year,1695,48,1647,0.028319


Risk difference: 0.3988
Risk ratio:      15.0819
Odds ratio:      25.5798


## Why these measures can tell different stories

Suppose:

\[
p_A=0.40,\qquad p_B=0.20.
\]

Then:

\[
RD=0.20,
\]

\[
RR=2,
\]

while:

\[
OR=
\frac{0.4/0.6}{0.2/0.8}
\approx 2.67.
\]

For non-rare outcomes, the odds ratio can look considerably more extreme than the risk ratio.

This matters because churn is not vanishingly rare.

# 8. Bootstrap confidence intervals for effect sizes

Analytical formulas are available for many simple estimators, but the bootstrap gives us a general computational framework.

For each bootstrap repetition:

1. sample \(n\) customers **with replacement**;
2. recompute \(RD\), \(RR\), and \(OR\);
3. use the empirical bootstrap distribution to estimate uncertainty.

Complexity:

\[
\Theta(Bn)
\]

for a straightforward implementation with \(B\) resamples.

In [11]:
class BootstrapInference:
    def __init__(
        self,
        n_bootstrap: int = 5_000,
        seed: int = 42,
    ) -> None:
        self.n_bootstrap = n_bootstrap
        self.rng = np.random.default_rng(seed)

    def contract_effects(
        self,
        data: pd.DataFrame,
        group_a: str = "Month-to-month",
        group_b: str = "Two year",
    ) -> pd.DataFrame:
        work = data[data["contract"].isin([group_a, group_b])].reset_index(drop=True)

        values = np.empty((self.n_bootstrap, 3))

        for b in range(self.n_bootstrap):
            idx = self.rng.integers(0, len(work), size=len(work))
            sample = work.iloc[idx]

            pa = sample.loc[sample["contract"] == group_a, "churn_num"].mean()
            pb = sample.loc[sample["contract"] == group_b, "churn_num"].mean()

            # Extremely unlikely empty bootstrap group guard.
            if not np.isfinite(pa) or not np.isfinite(pb) or pb == 0:
                values[b, :] = np.nan
                continue

            rd = pa - pb
            rr = pa / pb
            odds_a = pa / (1 - pa)
            odds_b = pb / (1 - pb)
            or_ = odds_a / odds_b

            values[b, :] = [rd, rr, or_]

        return pd.DataFrame(
            values,
            columns=["risk_difference", "risk_ratio", "odds_ratio"],
        ).dropna()


bootstrapper = BootstrapInference(
    n_bootstrap=CFG.bootstrap_reps,
    seed=RANDOM_STATE,
)

boot_effects = bootstrapper.contract_effects(df)

summary = pd.DataFrame({
    "estimate": [risk_difference, risk_ratio, odds_ratio],
    "bootstrap_se": [
        boot_effects["risk_difference"].std(ddof=1),
        boot_effects["risk_ratio"].std(ddof=1),
        boot_effects["odds_ratio"].std(ddof=1),
    ],
    "ci_low": [
        boot_effects["risk_difference"].quantile(0.025),
        boot_effects["risk_ratio"].quantile(0.025),
        boot_effects["odds_ratio"].quantile(0.025),
    ],
    "ci_high": [
        boot_effects["risk_difference"].quantile(0.975),
        boot_effects["risk_ratio"].quantile(0.975),
        boot_effects["odds_ratio"].quantile(0.975),
    ],
}, index=["Risk difference", "Risk ratio", "Odds ratio"])

display(summary)

,estimate,bootstrap_se,ci_low,ci_high
Risk difference,0.398778,0.008799,0.381639,0.415878
Risk ratio,15.081855,2.298935,11.631533,20.713907
Odds ratio,25.579814,4.073380,19.485748,35.456192


In [12]:
hist, edges = np.histogram(
    boot_effects["risk_difference"],
    bins=45,
)

p = figure(
    height=400,
    width=760,
    title="Bootstrap distribution: contract risk difference",
    x_axis_label="Month-to-month churn rate − two-year churn rate",
    y_axis_label="Bootstrap frequency",
)

p.quad(
    top=hist,
    bottom=0,
    left=edges[:-1],
    right=edges[1:],
    fill_alpha=0.5,
)

p.add_layout(Span(
    location=risk_difference,
    dimension="height",
    line_width=2,
))

for value in boot_effects["risk_difference"].quantile([0.025, 0.975]):
    p.add_layout(Span(
        location=float(value),
        dimension="height",
        line_dash="dashed",
        line_width=2,
    ))

show(p)

# 9. Chi-square test and Cramér's V

For `Contract × Churn`:

\[
H_0:
Contract \perp Churn
\]

versus

\[
H_1:
Contract \not\perp Churn.
\]

Pearson's chi-square statistic is:

\[
\chi^2
=
\sum_{i,j}
\frac{(O_{ij}-E_{ij})^2}{E_{ij}}.
\]

Because a p-value does not measure association magnitude, compute **Cramér's V**:

\[
V=
\sqrt{
\frac{\chi^2}
{n\min(r-1,c-1)}
}.
\]

In [13]:
def chi_square_with_cramers_v(
    data: pd.DataFrame,
    feature: str,
    target: str = "churn_num",
) -> dict:
    table = pd.crosstab(data[feature], data[target])

    chi2, p_value, dof, expected = stats.chi2_contingency(table)

    n = table.to_numpy().sum()
    r, c = table.shape
    denom = n * min(r - 1, c - 1)
    v = np.sqrt(chi2 / denom) if denom > 0 else np.nan

    return {
        "feature": feature,
        "chi2": chi2,
        "dof": dof,
        "p_value": p_value,
        "cramers_v": v,
        "min_expected": np.min(expected),
    }


contract_test = chi_square_with_cramers_v(df, "contract")
contract_test

{'feature': 'contract',
 'chi2': 1184.5965720837926,
 'dof': 2,
 'p_value': 5.863038300673391e-258,
 'cramers_v': 0.4101156965761409,
 'min_expected': 390.889819679114}

### Interpretation hierarchy

A strong statistical report should usually follow:

1. **effect size**;
2. **confidence interval / uncertainty**;
3. **hypothesis-test result**;
4. **assumptions and limitations**.

Not:

> p-value first, everything else optional.

# 10. Permutation test for the contract risk difference

The bootstrap asks:

> How variable is my estimator under repeated resampling from the empirical distribution?

A permutation test asks:

> What risk differences would occur if churn labels were exchangeable with respect to contract under the null?

We hold the contract labels fixed and shuffle `churn_num`.

In [14]:
work = df[df["contract"].isin(["Month-to-month", "Two year"])].copy()

contract_array = work["contract"].to_numpy()
y_array = work["churn_num"].to_numpy()

mask_a = contract_array == "Month-to-month"
mask_b = contract_array == "Two year"

perm_stats = np.empty(CFG.permutation_reps)

for b in range(CFG.permutation_reps):
    y_perm = rng.permutation(y_array)

    perm_stats[b] = (
        y_perm[mask_a].mean()
        - y_perm[mask_b].mean()
    )

perm_p = (
    1 + np.sum(np.abs(perm_stats) >= abs(risk_difference))
) / (CFG.permutation_reps + 1)

print(f"Observed risk difference: {risk_difference:.4f}")
print(f"Two-sided permutation p-value ≈ {perm_p:.6g}")

Observed risk difference: 0.3988
Two-sided permutation p-value ≈ 9.999e-05


In [15]:
hist, edges = np.histogram(perm_stats, bins=45)

p = figure(
    height=400,
    width=760,
    title="Permutation null distribution: contract risk difference",
    x_axis_label="Risk difference under shuffled churn labels",
    y_axis_label="Frequency",
)

p.quad(
    top=hist,
    bottom=0,
    left=edges[:-1],
    right=edges[1:],
    fill_alpha=0.5,
)

p.add_layout(Span(
    location=risk_difference,
    dimension="height",
    line_width=2,
))
p.add_layout(Span(
    location=-risk_difference,
    dimension="height",
    line_width=2,
))

show(p)

# 11. Continuous variables: Welch tests + standardized effect sizes

Consider `MonthlyCharges`.

A simple inferential question is:

\[
H_0:
\mu_{Churn}=\mu_{NoChurn}.
\]

Because equal variance need not hold, use **Welch's t-test**.

But once again, statistical significance does not tell us the magnitude.

We add **Hedges' \(g\)**, a bias-corrected standardized mean difference.

In [16]:
def hedges_g(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    nx, ny = len(x), len(y)
    vx = np.var(x, ddof=1)
    vy = np.var(y, ddof=1)

    pooled_sd = np.sqrt(
        ((nx - 1) * vx + (ny - 1) * vy)
        / (nx + ny - 2)
    )

    d = (np.mean(x) - np.mean(y)) / pooled_sd
    correction = 1 - 3 / (4 * (nx + ny) - 9)

    return correction * d


charges_churn = df.loc[
    df["churn_num"] == 1,
    "monthly_charges"
].dropna().to_numpy()

charges_stay = df.loc[
    df["churn_num"] == 0,
    "monthly_charges"
].dropna().to_numpy()

welch = stats.ttest_ind(
    charges_churn,
    charges_stay,
    equal_var=False,
)

g = hedges_g(charges_churn, charges_stay)

print(f"Mean charges — churners:     {charges_churn.mean():.2f}")
print(f"Mean charges — non-churners: {charges_stay.mean():.2f}")
print(f"Mean difference:             {charges_churn.mean() - charges_stay.mean():.2f}")
print(f"Welch t statistic:           {welch.statistic:.4f}")
print(f"p-value:                     {welch.pvalue:.6g}")
print(f"Hedges' g:                   {g:.4f}")

Mean charges — churners:     74.44
Mean charges — non-churners: 61.27
Mean difference:             13.18
Welch t statistic:           18.4075
p-value:                     8.59245e-73
Hedges' g:                   0.4462


## Why this is still only an unadjusted comparison

Monthly charges are related to service bundle and internet service.

Therefore:

\[
MonthlyCharges
\leftrightarrow
Services
\leftrightarrow
Churn
\]

may create confounding or proxy relationships.

The Welch test does not answer:

> Is monthly charge independently associated with churn after accounting for contract, tenure, internet service and support options?

For that, we need multivariable modelling.

# 12. Tenure as a potential confounder / structural variable

Long-tenure customers and new customers differ strongly in:

- exposure duration;
- contract composition;
- accumulated charges;
- observed churn behavior.

We inspect churn by `Contract × tenure band`.

In [17]:
contract_tenure = (
    df.groupby(
        ["contract", "tenure_band"],
        observed=True,
    )["churn_num"]
    .agg(["count", "mean"])
    .reset_index()
    .rename(columns={"mean": "churn_rate"})
)

display(contract_tenure)

,contract,tenure_band,count,churn_rate
0,Month-to-month,0–12,1994,0.513541
1,Month-to-month,13–24,737,0.377205
2,Month-to-month,25–48,802,0.329177
3,Month-to-month,49–60,234,0.277778
4,Month-to-month,61+,108,0.222222
5,One year,0–12,124,0.104839
6,One year,13–24,197,0.081218
7,One year,25–48,518,0.106178
8,One year,49–60,321,0.137072
9,One year,61+,313,0.121406


In [18]:
p = figure(
    x_range=[str(x) for x in df["tenure_band"].dropna().cat.categories],
    height=430,
    width=780,
    title="Churn by contract within tenure bands",
    x_axis_label="Tenure band (months)",
    y_axis_label="Observed churn proportion",
)

for contract in contract_tenure["contract"].dropna().unique():
    part = contract_tenure[contract_tenure["contract"] == contract]
    x = part["tenure_band"].astype(str).tolist()
    y = part["churn_rate"].tolist()

    p.line(
        x,
        y,
        line_width=2,
        legend_label=str(contract),
    )
    p.circle(x, y, size=7)

p.legend.click_policy = "hide"
show(p)

### Confounding intuition

Suppose an unadjusted contract difference is very large.

Part of that may reflect that contract types have different tenure distributions.

Adjusted regression asks a different conditional question:

\[
P(Y=1\mid Contract, Tenure, \ldots)
\]

rather than simply:

\[
P(Y=1\mid Contract).
\]

This is why an unadjusted odds ratio and an adjusted odds ratio can differ substantially.

# 13. Multiple hypothesis testing and false discoveries

If we independently test 20 unrelated null hypotheses at:

\[
\alpha=0.05,
\]

we should expect roughly one false rejection on average when all nulls are true.

We examine several categorical telecom-service variables, then apply the **Benjamini–Hochberg** procedure to control the False Discovery Rate (FDR).

BH works on ordered p-values:

\[
p_{(1)}
\le
p_{(2)}
\le
\cdots
\le
p_{(m)}.
\]

It is less conservative than family-wise-error methods such as Bonferroni when the goal is discovery-oriented screening.

In [19]:
categorical_tests = [
    "gender",
    "senior_citizen",
    "partner",
    "dependents",
    "phone_service",
    "multiple_lines",
    "internet_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
    "contract",
    "paperless_billing",
    "payment_method",
]

test_rows = [
    chi_square_with_cramers_v(df, feature)
    for feature in categorical_tests
]

multiple_test_df = pd.DataFrame(test_rows)

reject, p_adj, _, _ = multipletests(
    multiple_test_df["p_value"],
    alpha=0.05,
    method="fdr_bh",
)

multiple_test_df["p_fdr_bh"] = p_adj
multiple_test_df["reject_fdr_05"] = reject

multiple_test_df = multiple_test_df.sort_values(
    ["p_fdr_bh", "cramers_v"],
    ascending=[True, False],
).reset_index(drop=True)

display(multiple_test_df)

,feature,chi2,dof,p_value,cramers_v,min_expected,p_fdr_bh,reject_fdr_05
0,contract,1184.596572,2,5.863038e-258,0.410116,390.889820,9.380861e-257,True
1,online_security,849.998968,2,2.661150e-185,0.347400,404.954423,2.128920e-184,True
2,tech_support,828.197068,2,1.443084e-180,0.342916,404.954423,7.696448e-180,True
3,internet_service,732.309590,2,9.571788e-160,0.322455,404.954423,3.828715e-159,True
4,payment_method,648.142327,3,3.682355e-140,0.303359,403.892943,1.178353e-139,True
5,online_backup,601.812790,2,2.079759e-131,0.292316,404.954423,5.546025e-131,True
6,device_protection,558.419369,2,5.505219e-122,0.281580,404.954423,1.258336e-121,True
7,streaming_movies,375.661479,2,2.667757e-82,0.230951,404.954423,5.335514e-82,True
8,streaming_tv,374.203943,2,5.528994e-82,0.230502,404.954423,9.829324e-82,True
9,paperless_billing,258.277649,1,4.073355e-58,0.191498,762.142269,6.517367e-58,True


### Important caveat

FDR-adjusted bivariate screening is **not** a replacement for a well-specified multivariable model.

Why?

Because:

- correlated predictors share information;
- confounding can create marginal associations;
- adjustment can reveal suppressed associations;
- a non-significant marginal relationship can become relevant conditionally.

Use the table as **exploratory evidence**, not as a mechanical feature-selection algorithm.

# 14. Multicollinearity: why `TotalCharges` is tricky

Conceptually:

\[
TotalCharges
\approx
Tenure \times MonthlyCharges
\]

with billing-plan complications.

So including all three can create a strong structural dependency.

For inference, this can inflate standard errors and make individual coefficients unstable.

We inspect VIF:

\[
VIF_j=
\frac{1}{1-R_j^2}.
\]

In [20]:
vif_data = (
    df[[
        "tenure",
        "monthly_charges",
        "total_charges",
    ]]
    .dropna()
    .astype(float)
)

vif_X = sm.add_constant(vif_data)

vif_table = pd.DataFrame({
    "feature": vif_X.columns,
    "VIF": [
        variance_inflation_factor(vif_X.to_numpy(), i)
        for i in range(vif_X.shape[1])
    ],
})

display(vif_table)

,feature,VIF
0,const,14.973839
1,tenure,5.844646
2,monthly_charges,3.225293
3,total_charges,9.526697


## Modelling decision

The main inferential model excludes `TotalCharges`.

This is not because the variable is "bad."

It is because our inferential question is better served by keeping:

- **tenure** as customer-history duration;
- **monthly charges** as current price/bundle intensity;

rather than simultaneously estimating a third highly dependent cumulative billing measure.

For pure prediction, the trade-off can be different.

# 15. Adjusted logistic regression — the core inferential model

We model churn probability with a binomial GLM and logit link:

\[
\log
\frac{p_i}{1-p_i}
=
\beta_0
+
\beta_1 Contract_i
+
\beta_2 Tenure_i
+
\beta_3 MonthlyCharges_i
+\cdots
\]

Main covariates:

- contract;
- tenure;
- monthly charge;
- internet service;
- tech support;
- online security;
- payment method;
- paperless billing;
- senior-citizen indicator;
- partner;
- dependents.

### Why this is an inferential model

The focus is on:

\[
\hat\beta_j,\quad
SE(\hat\beta_j),\quad
CI_j,\quad
e^{\hat\beta_j}.
\]

We use **HC3 robust covariance estimates** to reduce sensitivity of standard errors to certain variance/misspecification issues.

Robust SEs do not fix a badly chosen mean structure or omitted confounding.

In [21]:
inferential_columns = [
    "churn_num",
    "contract",
    "tenure_years",
    "monthly_charges_10",
    "internet_service",
    "tech_support",
    "online_security",
    "payment_method",
    "paperless_billing",
    "senior_citizen",
    "partner",
    "dependents",
]

infer_df = df[inferential_columns].dropna().copy()

print("Rows available:", len(df))
print("Rows in inferential model:", len(infer_df))
print(f"Retention fraction: {len(infer_df) / len(df):.4f}")

Rows available: 7043
Rows in inferential model: 7043
Retention fraction: 1.0000


In [22]:
MAIN_FORMULA = '''
churn_num
~ C(contract, Treatment(reference="Two year"))
+ tenure_years
+ monthly_charges_10
+ C(internet_service, Treatment(reference="DSL"))
+ C(tech_support, Treatment(reference="No"))
+ C(online_security, Treatment(reference="No"))
+ C(payment_method, Treatment(reference="Electronic check"))
+ C(paperless_billing, Treatment(reference="No"))
+ senior_citizen
+ C(partner, Treatment(reference="No"))
+ C(dependents, Treatment(reference="No"))
'''

main_model = smf.glm(
    formula=MAIN_FORMULA,
    data=infer_df,
    family=sm.families.Binomial(),
).fit(cov_type="HC3")

print(main_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:              churn_num   No. Observations:                 7043
Model:                            GLM   Df Residuals:                     7027
Model Family:                Binomial   Df Model:                           15
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2954.1
Date:                Sat, 05 Sep 2026   Deviance:                       5908.2
Time:                        10:07:51   Pearson chi2:                 7.05e+03
No. Iterations:                     7   Pseudo R-squ. (CS):             0.2726
Covariance Type:                  HC3                                         
                                                                                              coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------

# 16. Interpret adjusted odds ratios

For coefficient \(\beta_j\):

\[
OR_j=e^{\beta_j}.
\]

A 95% CI for the odds ratio is:

\[
\left[
e^{L_j},
e^{U_j}
\right],
\]

where \([L_j,U_j]\) is the coefficient CI.

For continuous predictors:

- `tenure_years`: one-year increase;
- `monthly_charges_10`: $10 increase.

For categorical predictors, the OR is relative to the specified reference category.

In [23]:
def odds_ratio_table(result) -> pd.DataFrame:
    ci = result.conf_int()

    table = pd.DataFrame({
        "term": result.params.index,
        "coef_log_odds": result.params.values,
        "robust_se": result.bse.values,
        "odds_ratio": np.exp(result.params.values),
        "ci_low": np.exp(ci[0].values),
        "ci_high": np.exp(ci[1].values),
        "p_value": result.pvalues.values,
    })

    return table


or_table = odds_ratio_table(main_model)
display(or_table)

,term,coef_log_odds,robust_se,odds_ratio,ci_low,ci_high,p_value
0,Intercept,-1.951524,0.252951,0.142057,0.086527,0.233225,1.209454e-14
1,"C(contract, Treatment(reference=""Two year""))[T...",1.339008,0.167923,3.815256,2.745286,5.302246,1.536811e-15
2,"C(contract, Treatment(reference=""Two year""))[T...",0.677122,0.174935,1.968205,1.396900,2.773163,1.085216e-04
3,"C(internet_service, Treatment(reference=""DSL"")...",0.507461,0.139429,1.661068,1.263877,2.183081,2.731106e-04
4,"C(internet_service, Treatment(reference=""DSL"")...",-0.256112,1.628554,0.774055,0.031809,18.836480,8.750374e-01
5,"C(tech_support, Treatment(reference=""No""))[T.N...",-0.256112,2.941765,0.774055,0.002425,247.062609,9.306234e-01
6,"C(tech_support, Treatment(reference=""No""))[T.Yes]",-0.387176,0.087817,0.678971,0.571612,0.806494,1.038928e-05
7,"C(online_security, Treatment(reference=""No""))[...",-0.256112,1.311651,0.774055,0.059196,10.121662,8.451902e-01
8,"C(online_security, Treatment(reference=""No""))[...",-0.454871,0.085874,0.634530,0.536236,0.750841,1.177364e-07
9,"C(payment_method, Treatment(reference=""Electro...",-0.358951,0.093150,0.698409,0.581863,0.838299,1.164474e-04


In [24]:
plot_or = (
    or_table
    .query("term != 'Intercept'")
    .copy()
    .sort_values("odds_ratio")
)

plot_or["label"] = (
    plot_or["term"]
    .str.replace('C(contract, Treatment(reference="Two year"))[T.Month-to-month]', "Contract: month-to-month vs two-year", regex=False)
    .str.replace('C(contract, Treatment(reference="Two year"))[T.One year]', "Contract: one-year vs two-year", regex=False)
    .str.replace('C(internet_service, Treatment(reference="DSL"))[T.Fiber optic]', "Internet: fiber vs DSL", regex=False)
    .str.replace('C(internet_service, Treatment(reference="DSL"))[T.No]', "Internet: none vs DSL", regex=False)
    .str.replace('C(paperless_billing, Treatment(reference="No"))[T.Yes]', "Paperless billing: yes vs no", regex=False)
    .str.replace('C(partner, Treatment(reference="No"))[T.Yes]', "Partner: yes vs no", regex=False)
    .str.replace('C(dependents, Treatment(reference="No"))[T.Yes]', "Dependents: yes vs no", regex=False)
    .str.replace("tenure_years", "Tenure: +1 year", regex=False)
    .str.replace("monthly_charges_10", "Monthly charges: +$10", regex=False)
)

source = ColumnDataSource(plot_or)

p = figure(
    y_range=plot_or["label"].tolist(),
    height=max(520, 28 * len(plot_or)),
    width=950,
    x_axis_type="log",
    title="Adjusted odds ratios with robust 95% confidence intervals",
    x_axis_label="Odds ratio (log scale)",
)

p.circle(
    x="odds_ratio",
    y="label",
    size=8,
    source=source,
)

p.add_layout(Whisker(
    base="label",
    lower="ci_low",
    upper="ci_high",
    source=source,
    dimension="width",
))

p.add_layout(Span(
    location=1,
    dimension="height",
    line_dash="dashed",
    line_width=2,
))

p.add_tools(HoverTool(
    tooltips=[
        ("Term", "@label"),
        ("OR", "@odds_ratio{0.000}"),
        ("95% CI", "[@ci_low{0.000}, @ci_high{0.000}]"),
        ("p", "@p_value{0.000000}"),
    ]
))

show(p)

## How to interpret one row

Suppose the fitted output gave:

\[
OR_{\text{month-to-month}}=4.0.
\]

A correct statement would be:

> Conditional on the other measured covariates included in the model, month-to-month customers have approximately four times the estimated odds of churn relative to two-year customers.

A stronger statement such as:

> Moving a customer onto a two-year contract will reduce churn by 75%.

is **not justified** by this observational regression alone.

# 17. Unadjusted versus adjusted association

Now compare:

1. unadjusted month-to-month vs two-year OR;
2. adjusted OR from the multivariable GLM.

If they differ, possible explanations include:

- confounding;
- non-collapsibility of the odds ratio;
- correlated covariates;
- model specification.

The odds ratio is **non-collapsible**, so a difference between marginal and conditional ORs does not automatically prove confounding.

In [25]:
contract_term = 'C(contract, Treatment(reference="Two year"))[T.Month-to-month]'

adjusted_or = np.exp(main_model.params[contract_term])
adjusted_ci = np.exp(main_model.conf_int().loc[contract_term])

comparison = pd.DataFrame({
    "measure": ["Unadjusted OR", "Adjusted OR"],
    "estimate": [odds_ratio, adjusted_or],
    "ci_low": [
        summary.loc["Odds ratio", "ci_low"],
        adjusted_ci.iloc[0],
    ],
    "ci_high": [
        summary.loc["Odds ratio", "ci_high"],
        adjusted_ci.iloc[1],
    ],
})

display(comparison)

,measure,estimate,ci_low,ci_high
0,Unadjusted OR,25.579814,19.485748,35.456192
1,Adjusted OR,3.815256,2.745286,5.302246


# 18. Model-standardized adjusted churn probabilities

Odds ratios are mathematically convenient but often difficult for stakeholders.

A more intuitive model-based summary is:

> What average predicted churn probability would the fitted model assign if every row were evaluated under each contract category, while retaining that customer's other observed covariates?

For contract \(c\):

\[
\hat P_c
=
\frac{1}{n}
\sum_i
\hat p(X_i, Contract=c).
\]

This is sometimes called **standardization** or **g-computation-style prediction**.

Again, without causal assumptions we report it as a **model-standardized association**, not an intervention effect.

In [26]:
def standardized_contract_probabilities(
    result,
    data: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for contract in ["Month-to-month", "One year", "Two year"]:
        counterfactual_like = data.copy()
        counterfactual_like["contract"] = contract

        pred = result.predict(counterfactual_like)

        rows.append({
            "contract": contract,
            "standardized_churn_probability": pred.mean(),
        })

    return pd.DataFrame(rows)


standardized_contract = standardized_contract_probabilities(
    main_model,
    infer_df,
)

display(standardized_contract)

,contract,standardized_churn_probability
0,Month-to-month,0.297254
1,One year,0.201442
2,Two year,0.125837


In [27]:
source = ColumnDataSource(standardized_contract)

p = figure(
    x_range=standardized_contract["contract"].tolist(),
    height=390,
    width=650,
    title="Model-standardized churn probability by contract",
    y_axis_label="Average model-predicted churn probability",
    toolbar_location=None,
)

p.vbar(
    x="contract",
    top="standardized_churn_probability",
    width=0.65,
    source=source,
)

p.y_range.start = 0
show(p)

# 19. Bootstrap the adjusted standardized contrast

We can also bootstrap the **entire modelling procedure**.

For each bootstrap sample:

1. resample customers;
2. refit the GLM;
3. standardize predictions under:
   - month-to-month;
   - two-year;
4. record the difference.

This propagates model-estimation uncertainty into an interpretable probability-scale contrast.

This is computationally much more expensive than bootstrapping a simple mean:

\[
\Theta(B \times \text{model fit cost}).
\]

The default uses only 500 model-refit bootstrap repetitions. Increase it for a final study if runtime allows.

In [28]:
def bootstrap_standardized_contract_difference(
    data: pd.DataFrame,
    formula: str,
    n_bootstrap: int = 500,
    seed: int = 42,
) -> np.ndarray:
    local_rng = np.random.default_rng(seed)
    estimates = []

    for b in range(n_bootstrap):
        idx = local_rng.integers(0, len(data), size=len(data))
        sample = data.iloc[idx].copy()

        try:
            fit = smf.glm(
                formula=formula,
                data=sample,
                family=sm.families.Binomial(),
            ).fit(disp=0)

            m2m = sample.copy()
            two = sample.copy()

            m2m["contract"] = "Month-to-month"
            two["contract"] = "Two year"

            diff = fit.predict(m2m).mean() - fit.predict(two).mean()
            estimates.append(diff)

        except Exception:
            # Rare failed bootstrap fits are skipped.
            continue

    return np.asarray(estimates)


adjusted_boot = bootstrap_standardized_contract_difference(
    infer_df,
    MAIN_FORMULA,
    n_bootstrap=CFG.adjusted_bootstrap_reps,
    seed=RANDOM_STATE,
)

print("Successful bootstrap fits:", len(adjusted_boot))
print("Mean adjusted standardized difference:", adjusted_boot.mean())
print("95% percentile CI:", np.quantile(adjusted_boot, [0.025, 0.975]))

Successful bootstrap fits: 500
Mean adjusted standardized difference: 0.17063839140946002
95% percentile CI: [0.13347187 0.20448381]


# 20. Average marginal effect for a continuous predictor

For logistic regression:

\[
p_i=\sigma(\eta_i)
\]

and continuous \(x_j\),

\[
\frac{\partial p_i}{\partial x_{ij}}
=
p_i(1-p_i)\beta_j.
\]

For `monthly_charges_10`, the predictor unit is **$10**.

The Average Marginal Effect is:

\[
AME_j
=
\frac{1}{n}
\sum_i
p_i(1-p_i)\beta_j.
\]

This expresses the model's local probability-scale association much more directly than log odds.

In [29]:
fitted_p = main_model.predict(infer_df)

beta_charge = main_model.params["monthly_charges_10"]
ame_charge = np.mean(
    fitted_p * (1 - fitted_p) * beta_charge
)

beta_tenure = main_model.params["tenure_years"]
ame_tenure = np.mean(
    fitted_p * (1 - fitted_p) * beta_tenure
)

print(f"AME for +$10 monthly charge: {ame_charge:.5f}")
print(f"AME for +1 year tenure:      {ame_tenure:.5f}")

AME for +$10 monthly charge: 0.01497
AME for +1 year tenure:      -0.05165


### Interpretation caution

The AME is a derivative of the **fitted associational model**.

It does not automatically mean:

> increasing someone's bill by $10 will cause their churn probability to change by exactly the AME.

For causal interpretation, we would need a causal estimand and stronger assumptions/design.

# 21. Interaction / effect modification

The association between contract and churn may depend on tenure.

A model with interaction is:

\[
logit(p)
=
\cdots
+
\beta_1 Contract
+
\beta_2 Tenure
+
\beta_3(Contract\times Tenure).
\]

Then the contract contrast is **not constant** across tenure.

We compare reduced and interaction models using a likelihood-ratio test:

\[
LR=2(\ell_{full}-\ell_{reduced}).
\]

For this likelihood-based comparison, we fit the two models with ordinary likelihood covariance; the robust covariance is for coefficient inference, not the LR statistic itself.

In [30]:
INTERACTION_FORMULA = '''
churn_num
~ C(contract, Treatment(reference="Two year")) * tenure_years
+ monthly_charges_10
+ C(internet_service, Treatment(reference="DSL"))
+ C(tech_support, Treatment(reference="No"))
+ C(online_security, Treatment(reference="No"))
+ C(payment_method, Treatment(reference="Electronic check"))
+ C(paperless_billing, Treatment(reference="No"))
+ senior_citizen
+ C(partner, Treatment(reference="No"))
+ C(dependents, Treatment(reference="No"))
'''

reduced_lr = smf.glm(
    MAIN_FORMULA,
    data=infer_df,
    family=sm.families.Binomial(),
).fit()

interaction_lr = smf.glm(
    INTERACTION_FORMULA,
    data=infer_df,
    family=sm.families.Binomial(),
).fit()

lr_stat = 2 * (interaction_lr.llf - reduced_lr.llf)
df_diff = int(interaction_lr.df_model - reduced_lr.df_model)
lr_p = stats.chi2.sf(lr_stat, df=df_diff)

print(f"LR statistic: {lr_stat:.4f}")
print(f"df difference: {df_diff}")
print(f"p-value: {lr_p:.6g}")

LR statistic: 38.6137
df difference: 2
p-value: 4.12225e-09


## What interaction changes conceptually

Without interaction we might say:

> month-to-month is associated with higher churn odds than two-year, adjusting for tenure.

With interaction we instead ask:

> how does the month-to-month versus two-year contrast vary with tenure?

This is **effect modification** on the model scale.

# 22. Visualize the interaction on the probability scale

We create representative profiles while varying:

- tenure;
- contract.

Other covariates are held at reasonable reference/modal values.

This plot is for **model interpretation**, not causal simulation.

In [36]:


# --- USER: ensure these exist in your environment ---
# infer_df : pandas DataFrame with your data
# INTERACTION_FORMULA : string formula used for the GLM, e.g. "churn ~ tenure_years * C(contract) + monthly_charges_10 + C(internet_service) + ..."
# If INTERACTION_FORMULA is not defined, set it here:
# INTERACTION_FORMULA = "churn ~ tenure_years * C(contract) + monthly_charges_10 + C(internet_service) + tech_support + online_security + payment_method + paperless_billing + senior_citizen + partner + dependents"

# Quick guard
try:
    INTERACTION_FORMULA
except NameError:
    raise RuntimeError("Please define INTERACTION_FORMULA before running this script.")

# --- helper functions ---
def compute_design_and_vif(formula, df):
    """Return y, X (dataframes), rank, and VIF series for X."""
    y, X = patsy.dmatrices(formula, data=df, return_type="dataframe")
    rank = np.linalg.matrix_rank(X.values)
    shape = X.shape
    # compute VIFs (skip intercept column if present)
    vif_index = X.columns
    vif_vals = []
    for i in range(X.shape[1]):
        try:
            vif_vals.append(variance_inflation_factor(X.values, i))
        except Exception:
            vif_vals.append(np.nan)
    vif = pd.Series(vif_vals, index=vif_index)
    return y, X, rank, shape, vif

def try_fit_glm(formula, df, cov_type=None, regularized=False, reg_alpha=1.0, reg_L1_wt=0.0):
    """Try to fit GLM; return result object and a flag whether robust cov is attached."""
    model = smf.glm(formula, data=df, family=sm.families.Binomial())
    if regularized:
        # fit_regularized returns an object with params but not full results summary_frame
        res = model.fit_regularized(alpha=reg_alpha, L1_wt=reg_L1_wt, maxiter=10000)
        return res, False
    else:
        if cov_type is None:
            res = model.fit()
            return res, False
        else:
            # try to fit with cov_type passed to fit (may trigger robust cov computation)
            try:
                res = model.fit(cov_type=cov_type)
                return res, True
            except LinAlgError:
                # bubble up to caller
                raise
            except Exception as e:
                # other exceptions: return None
                raise

# --- Step 1: design matrix diagnostics ---
print("Building design matrix and computing diagnostics...")
y, X, rank, shape, vif = compute_design_and_vif(INTERACTION_FORMULA, infer_df)
print(f"Design matrix shape: {shape}, rank: {rank}")
if rank < shape[1]:
    print("WARNING: design matrix is rank-deficient (rank < ncols).")
else:
    print("Design matrix appears full rank.")

print("\nTop VIFs (descending):")
print(vif.sort_values(ascending=False).head(20))

# show any constant columns
const_cols = X.columns[X.var() == 0]
if len(const_cols) > 0:
    print("\nConstant columns detected (zero variance):", list(const_cols))

# --- Step 2: attempt original fit with HC3 (catch singularity) ---
interaction_model = None
robust_attached = False
fit_path = None

print("\nAttempting to fit GLM with cov_type='HC3' (this may raise LinAlgError if singular)...")
try:
    interaction_model, robust_attached = try_fit_glm(INTERACTION_FORMULA, infer_df, cov_type="HC3")
    fit_path = "hc3_direct"
    print("Success: fitted with cov_type='HC3' directly.")
except LinAlgError:
    print("LinAlgError during direct HC3 fit (singular matrix). Will try safer alternatives.")
except Exception as e:
    print("Error during direct HC3 fit:", type(e).__name__, str(e))
    print("Will try safer alternatives.")

# --- Step 3: fallback sequence if direct HC3 failed ---
if interaction_model is None:
    # 3a: fit without robust cov, then attach robust cov afterwards
    print("\nFallback 1: fit without robust cov, then compute robust covariance separately.")
    try:
        base_res = smf.glm(INTERACTION_FORMULA, data=infer_df, family=sm.families.Binomial()).fit()
        try:
            robust_res = base_res.get_robustcov_results(cov_type="HC3")
            interaction_model = robust_res
            robust_attached = True
            fit_path = "fit_then_get_robust"
            print("Success: base fit succeeded and HC3 robust covariance attached.")
        except LinAlgError:
            print("LinAlgError when computing robust covariance from base fit.")
            interaction_model = base_res  # keep base fit for predictions
            robust_attached = False
            fit_path = "fit_no_robust"
    except Exception as e:
        print("Base fit failed:", type(e).__name__, str(e))
        interaction_model = None

# 3b: if still failing or if VIFs indicate severe collinearity, center continuous predictors and refit
if interaction_model is None or (not robust_attached and fit_path == "fit_no_robust" and rank < shape[1]):
    print("\nFallback 2: centering continuous variables and refitting.")
    # choose continuous columns to center heuristically
    cont_candidates = ["tenure_years", "monthly_charges_10", "monthly_charges", "tenure"]
    to_center = [c for c in cont_candidates if c in infer_df.columns]
    if not to_center:
        # fallback: try to detect numeric columns used in formula via X.dtypes
        numeric_cols = [c for c in X.columns if np.issubdtype(X[c].dtype, np.number)]
        # exclude intercept
        numeric_cols = [c for c in numeric_cols if c.lower() != "intercept"]
        to_center = numeric_cols[:2]  # center up to two numeric columns
    print("Centering columns:", to_center)
    for c in to_center:
        infer_df[c + "_c"] = infer_df[c] - infer_df[c].mean()
    # create a modified formula by replacing occurrences of the original names with centered names
    formula_centered = INTERACTION_FORMULA
    for c in to_center:
        formula_centered = formula_centered.replace(c, c + "_c")
    print("Using centered formula:", formula_centered)
    try:
        base_res2 = smf.glm(formula_centered, data=infer_df, family=sm.families.Binomial()).fit()
        try:
            robust_res2 = base_res2.get_robustcov_results(cov_type="HC3")
            interaction_model = robust_res2
            robust_attached = True
            fit_path = "centered_fit_then_robust"
            print("Success: centered fit + HC3 attached.")
        except LinAlgError:
            print("LinAlgError when computing robust cov after centering.")
            interaction_model = base_res2
            robust_attached = False
            fit_path = "centered_fit_no_robust"
    except Exception as e:
        print("Centered fit failed:", type(e).__name__, str(e))
        interaction_model = None

# 3c: final fallback: regularized fit (L2) to get stable coefficients
if interaction_model is None:
    print("\nFallback 3: regularized GLM (L2) as last resort.")
    try:
        reg_res = smf.glm(INTERACTION_FORMULA, data=infer_df, family=sm.families.Binomial()).fit_regularized(alpha=1.0, L1_wt=0.0, maxiter=10000)
        interaction_model = reg_res
        robust_attached = False
        fit_path = "regularized"
        print("Success: regularized fit completed. Note: regularized result may not support get_prediction.summary_frame for CIs.")
    except Exception as e:
        print("Regularized fit failed:", type(e).__name__, str(e))
        raise RuntimeError("All fitting attempts failed. Inspect design matrix and formula for collinearity or separation.")

print(f"\nFinal fit path used: {fit_path}")
print("Model params (head):")
try:
    print(interaction_model.params.head(10))
except Exception:
    # fit_regularized returns an array-like; try to print
    try:
        print(pd.Series(interaction_model.params).head(10))
    except Exception:
        print("Could not display params.")

# --- Step 4: build prediction profile grid (same approach as your earlier code) ---
print("\nBuilding prediction profile grid and computing predictions...")

tenure_grid = np.linspace(infer_df["tenure_years"].min(), infer_df["tenure_years"].max(), 120)

profile_rows = []
for contract in ["Month-to-month", "One year", "Two year"]:
    for tenure_years in tenure_grid:
        profile_rows.append({
            "contract": contract,
            "tenure_years": tenure_years,
            "monthly_charges_10": infer_df["monthly_charges_10"].median() if "monthly_charges_10" in infer_df.columns else infer_df.select_dtypes("number").median().iloc[0],
            "internet_service": "DSL" if "internet_service" in infer_df.columns else infer_df.select_dtypes("object").columns[0],
            "tech_support": "No" if "tech_support" in infer_df.columns else "No",
            "online_security": "No" if "online_security" in infer_df.columns else "No",
            "payment_method": "Electronic check" if "payment_method" in infer_df.columns else "Unknown",
            "paperless_billing": "Yes" if "paperless_billing" in infer_df.columns else "No",
            "senior_citizen": 0 if "senior_citizen" in infer_df.columns else 0,
            "partner": "No" if "partner" in infer_df.columns else "No",
            "dependents": "No" if "dependents" in infer_df.columns else "No",
        })

profile = pd.DataFrame(profile_rows)

# If we used centered formula, ensure profile has centered columns too
if fit_path and "centered" in fit_path:
    # detect which centered columns were created
    centered_cols = [c for c in infer_df.columns if c.endswith("_c")]
    for c in centered_cols:
        orig = c[:-2]
        if orig in profile.columns:
            profile[c] = profile[orig] - infer_df[orig].mean()
        else:
            # if original not in profile (unlikely), set to mean 0
            profile[c] = 0.0

# Predictions and CIs
ci_available = False
try:
    # prefer get_prediction on statsmodels results (works for non-regularized GLMResults)
    pred = interaction_model.get_prediction(profile)
    pred_df = pred.summary_frame(alpha=0.05)
    profile["predicted_churn"] = pred_df["mean"]
    # summary_frame may return mean_ci_lower/upper on link scale or response scale depending on model; GLM.get_prediction returns on response by default
    if "mean_ci_lower" in pred_df.columns and "mean_ci_upper" in pred_df.columns:
        profile["ci_lower"] = pred_df["mean_ci_lower"]
        profile["ci_upper"] = pred_df["mean_ci_upper"]
        ci_available = True
    else:
        # fallback: use mean +/- 1.96*se_mean
        if "mean_se" in pred_df.columns:
            profile["ci_lower"] = pred_df["mean"] - 1.96 * pred_df["mean_se"]
            profile["ci_upper"] = pred_df["mean"] + 1.96 * pred_df["mean_se"]
            ci_available = True
except Exception as e:
    # regularized results or some result types may not support get_prediction
    print("Could not compute prediction CIs via get_prediction:", type(e).__name__, str(e))
    try:
        # try plain predict (no CI)
        profile["predicted_churn"] = interaction_model.predict(profile)
    except Exception as e2:
        # if predict fails, try using model's predict via formula/model object
        try:
            model_obj = smf.glm(INTERACTION_FORMULA, data=infer_df, family=sm.families.Binomial())
            profile["predicted_churn"] = model_obj.fit().predict(profile)
        except Exception as e3:
            raise RuntimeError("Prediction failed with all fallbacks.") from e3

# Clip probabilities to [0,1]
profile["predicted_churn"] = profile["predicted_churn"].clip(0, 1)
if ci_available:
    profile["ci_lower"] = profile["ci_lower"].clip(0, 1)
    profile["ci_upper"] = profile["ci_upper"].clip(0, 1)

# --- Step 5: Bokeh plot with lines and optional CI bands ---
print("Rendering Bokeh plot (predictions across tenure by contract)...")
p = figure(
    height=450,
    width=780,
    title="Adjusted model: churn probability across tenure by contract",
    x_axis_label="Tenure (years)",
    y_axis_label="Predicted churn probability",
    tools="hover,save,reset",
)

palette = Category10[3]
contracts = ["Month-to-month", "One year", "Two year"]

for i, contract in enumerate(contracts):
    part = profile[profile["contract"] == contract].sort_values("tenure_years")
    src = ColumnDataSource(part)
    p.line(
        x="tenure_years",
        y="predicted_churn",
        source=src,
        line_width=2,
        color=palette[i],
        legend_label=contract,
    )
    if ci_available:
        p.varea(
            x="tenure_years",
            y1="ci_lower",
            y2="ci_upper",
            source=src,
            fill_alpha=0.15,
            fill_color=palette[i],
        )

hover = p.select_one(HoverTool)
hover.tooltips = [
    ("Contract", "@contract"),
    ("Tenure (years)", "@tenure_years{0.00}"),
    ("Predicted churn", "@predicted_churn{0.000}"),
]
if ci_available:
    hover.tooltips.append(("95% CI", "@ci_lower{0.000} — @ci_upper{0.000}"))
hover.mode = "vline"

p.legend.click_policy = "hide"
p.y_range.start = 0
p.y_range.end = min(1.0, (profile["ci_upper"].max() if ci_available else profile["predicted_churn"].max()) * 1.05)

show(p)

print("\nDone. Summary:")
print(f" - Fit path: {fit_path}")
print(f" - Robust covariance attached: {robust_attached}")
print(f" - CI available for predictions: {ci_available}")


Building design matrix and computing diagnostics...
Design matrix shape: (7043, 20), rank: 18

Top VIFs (descending):
C(internet_service, Treatment(reference="DSL"))[T.No]                                            inf
C(tech_support, Treatment(reference="No"))[T.No internet service]                                inf
C(online_security, Treatment(reference="No"))[T.No internet service]                             inf
Intercept                                                                                  82.653215
C(contract, Treatment(reference="Two year"))[T.Month-to-month]                             13.267221
C(contract, Treatment(reference="Two year"))[T.One year]                                   12.186683
C(contract, Treatment(reference="Two year"))[T.One year]:tenure_years                       9.560595
monthly_charges_10                                                                          9.210576
tenure_years                                                              


Done. Summary:
 - Fit path: regularized
 - Robust covariance attached: False
 - CI available for predictions: False


# 23. Is tenure linear on the logit scale?

A standard logistic model assumes:

\[
logit(p)
=
\beta_0+\beta_1 Tenure+\cdots
\]

which means **linearity in the logit**, not linearity in probability.

We test a quadratic extension:

\[
logit(p)
=
\beta_0+
\beta_1 Tenure+
\beta_2 Tenure^2+\cdots.
\]

A significant quadratic term suggests the linear-logit assumption may be too restrictive.

In production work, splines are often preferable to arbitrary high-order polynomials.

In [37]:
infer_df = infer_df.copy()
infer_df["tenure_years_sq"] = infer_df["tenure_years"] ** 2

QUADRATIC_FORMULA = MAIN_FORMULA + " + tenure_years_sq"

linear_fit = smf.glm(
    MAIN_FORMULA,
    data=infer_df,
    family=sm.families.Binomial(),
).fit()

quadratic_fit = smf.glm(
    QUADRATIC_FORMULA,
    data=infer_df,
    family=sm.families.Binomial(),
).fit()

lr_stat_quad = 2 * (quadratic_fit.llf - linear_fit.llf)
df_diff_quad = int(quadratic_fit.df_model - linear_fit.df_model)
lr_p_quad = stats.chi2.sf(lr_stat_quad, df=df_diff_quad)

print(f"Quadratic-vs-linear LR statistic: {lr_stat_quad:.4f}")
print(f"df difference: {df_diff_quad}")
print(f"p-value: {lr_p_quad:.6g}")

Quadratic-vs-linear LR statistic: 56.8248
df difference: 1
p-value: 4.76417e-14


# 24. Influence diagnostics

A fitted regression can be disproportionately affected by unusual observations.

For GLMs, influence diagnostics can include:

- leverage;
- standardized residuals;
- Cook's distance.

An influential customer is not automatically a "bad row."

It may represent:

- genuine rare behavior;
- data-entry error;
- a subgroup the model handles poorly.

Deletion should require substantive justification.

In [38]:
diagnostic_fit = smf.glm(
    MAIN_FORMULA,
    data=infer_df,
    family=sm.families.Binomial(),
).fit()

influence = diagnostic_fit.get_influence(observed=True)
influence_frame = influence.summary_frame()

display(
    influence_frame
    .assign(row_index=infer_df.index.to_numpy())
    .sort_values("cooks_d", ascending=False)
    .head(12)
)

,dfb_Intercept,"dfb_C(contract, Treatment(reference=""Two year""))[T.Month-to-month]","dfb_C(contract, Treatment(reference=""Two year""))[T.One year]","dfb_C(internet_service, Treatment(reference=""DSL""))[T.Fiber optic]","dfb_C(internet_service, Treatment(reference=""DSL""))[T.No]","dfb_C(tech_support, Treatment(reference=""No""))[T.No internet service]","dfb_C(tech_support, Treatment(reference=""No""))[T.Yes]","dfb_C(online_security, Treatment(reference=""No""))[T.No internet service]","dfb_C(online_security, Treatment(reference=""No""))[T.Yes]","dfb_C(payment_method, Treatment(reference=""Electronic check""))[T.Bank transfer (automatic)]",...,"dfb_C(partner, Treatment(reference=""No""))[T.Yes]","dfb_C(dependents, Treatment(reference=""No""))[T.Yes]",dfb_tenure_years,dfb_monthly_charges_10,dfb_senior_citizen,cooks_d,standard_resid,hat_diag,dffits_internal,row_index
4513,0.034341,-0.097891,-0.115831,-0.075192,0.017576,0.017576,0.007333,0.017576,0.017361,-0.021040,...,-0.013441,0.055557,0.006648,0.061288,0.083698,0.002547,9.084076,0.000555,0.214122,4513
4819,0.103174,-0.114264,-0.118501,0.024258,0.035551,0.035551,-0.015272,0.035551,-0.014417,0.043572,...,-0.071229,0.069517,0.041497,-0.034600,0.010313,0.002465,10.569245,0.000397,0.210661,4819
3779,0.061425,-0.104649,-0.117306,-0.041495,-0.004484,-0.004484,0.020067,-0.004484,0.026411,-0.020480,...,-0.009755,0.055024,0.005681,0.022674,0.084818,0.002338,8.877424,0.000534,0.205133,3779
6813,0.145141,-0.115897,-0.118805,0.044103,-0.060886,-0.060886,0.050568,-0.060886,0.049035,-0.051641,...,0.037636,-0.047429,0.045144,-0.081894,-0.018485,0.002313,8.466595,0.000580,0.204037,6813
4272,0.086776,-0.117511,-0.120166,0.006772,0.050969,0.050969,-0.020564,0.050969,-0.012972,-0.019350,...,0.043784,-0.049659,0.018474,-0.023144,-0.018743,0.002282,8.435280,0.000577,0.202651,4272
4698,0.052831,-0.121502,-0.123881,-0.089066,-0.000470,-0.000470,-0.064228,-0.000470,-0.052729,0.052512,...,0.023210,-0.028116,0.001683,0.060705,0.054950,0.002242,4.213029,0.002269,0.200890,4698
4149,0.134764,-0.123125,-0.124265,-0.002840,-0.062972,-0.062972,-0.038096,-0.062972,-0.037599,-0.045557,...,0.033906,-0.044746,0.063244,-0.038324,-0.027769,0.002211,6.154749,0.001049,0.199484,4149
268,0.101055,-0.114427,-0.118942,0.028342,0.025260,0.025260,-0.016815,0.025260,-0.014462,-0.023512,...,-0.032333,-0.019872,0.067791,-0.036320,-0.006177,0.002162,10.743043,0.000337,0.197294,268
6724,0.079518,-0.114277,-0.119472,-0.020255,-0.013662,-0.013662,0.029065,-0.013662,0.037891,-0.016213,...,-0.046897,0.000447,0.017993,-0.008566,0.073414,0.002145,6.890463,0.000813,0.196507,6724
3971,0.104179,-0.135693,-0.126221,0.000223,0.055315,0.055315,-0.021278,0.055315,-0.011558,-0.013734,...,-0.017866,-0.021905,-0.028132,0.000989,-0.000382,0.002107,6.286651,0.000959,0.194764,3971


# 25. Model adequacy: calibration by fitted-risk groups

Statistically significant coefficients do not guarantee a good probability model.

We divide customers into fitted-risk bins and compare:

\[
\text{mean predicted churn}
\]

with

\[
\text{observed churn rate}.
\]

Perfect agreement would fall near the 45-degree line.

In [39]:
infer_pred = main_model.predict(infer_df)

calibration_infer = pd.DataFrame({
    "y": infer_df["churn_num"].to_numpy(),
    "p": infer_pred.to_numpy(),
})

calibration_infer["bin"] = pd.qcut(
    calibration_infer["p"],
    q=10,
    duplicates="drop",
)

calibration_summary = (
    calibration_infer.groupby("bin", observed=True)
    .agg(
        n=("y", "size"),
        mean_pred=("p", "mean"),
        observed_rate=("y", "mean"),
    )
    .reset_index()
)

display(calibration_summary)

,bin,n,mean_pred,observed_rate
0,"(0.00353, 0.0155]",705,0.009897,0.012766
1,"(0.0155, 0.0332]",704,0.024049,0.022727
2,"(0.0332, 0.0664]",704,0.047896,0.048295
3,"(0.0664, 0.119]",704,0.090271,0.098011
4,"(0.119, 0.182]",705,0.151572,0.163121
5,"(0.182, 0.275]",704,0.226108,0.224432
6,"(0.275, 0.391]",704,0.332711,0.319602
7,"(0.391, 0.522]",704,0.455601,0.441761
8,"(0.522, 0.655]",704,0.589473,0.571023
9,"(0.655, 0.832]",705,0.725992,0.751773


In [40]:
p = figure(
    height=450,
    width=650,
    title="Inferential GLM: observed vs fitted churn",
    x_axis_label="Mean fitted probability",
    y_axis_label="Observed churn rate",
    x_range=(0, 1),
    y_range=(0, 1),
)

p.line([0, 1], [0, 1], line_dash="dashed")
p.circle(
    calibration_summary["mean_pred"],
    calibration_summary["observed_rate"],
    size=9,
)

show(p)

# 26. Inferential assumptions checklist

Before writing conclusions, inspect the assumptions behind the model.

## A. Independent observational units

We treat rows as independent customers.

Repeated observations from the same household/account would violate this.

---

## B. Correctly specified conditional mean

We assume the chosen logit structure is adequate.

Potential violations:

- nonlinear tenure relationship;
- omitted interactions;
- omitted important customer segments.

---

## C. No perfect or quasi-separation

A predictor category that almost perfectly determines churn can create unstable logistic estimates.

---

## D. Positivity / overlap for standardized comparisons

For meaningful adjusted comparisons, we need reasonable representation of covariate profiles across contract types.

If some customer profiles occur only in one contract type, extrapolation becomes stronger.

---

## E. Missing-data assumptions

Complete-case inference can be distorted if missingness is informative.

---

## F. Measurement validity

Variables such as "TechSupport = No" are account-state labels, not randomized treatments.

---

## G. Causal interpretation requires much more

To interpret an association as causal, we would need assumptions about:

- no unmeasured confounding;
- consistency;
- positivity;
- temporal ordering;
- selection mechanisms.

This notebook **does not make causal claims**.

# 27. Inferential analytics: what a final write-up should look like

A high-quality conclusion should contain:

### 1. Population / data scope

> The analysis concerns customers represented by this telecom sample or a sufficiently comparable customer population.

### 2. Effect magnitude

Report:

- risk difference;
- risk ratio;
- odds ratio;
- standardized probability contrast where useful.

### 3. Uncertainty

Report:

- robust standard errors;
- 95% confidence intervals;
- bootstrap intervals.

### 4. Hypothesis evidence

Use p-values as supporting evidence, not as the entire conclusion.

### 5. Adjustment

State exactly which measured variables were controlled for.

### 6. Model diagnostics

Mention:

- nonlinearity;
- multicollinearity;
- calibration;
- influential observations.

### 7. Limitations

Most importantly:

> The dataset is observational, so adjusted associations should not be described automatically as causal effects.

# Part II — Predictive Analytics

We now **change the objective**.

Instead of estimating interpretable population relationships, we ask:

\[
\boxed{
\text{Can we estimate churn risk for unseen customers?}
}
\]

The same data can support both goals, but the validation framework changes.

For prediction:

- split before fitting preprocessing;
- use cross-validation;
- use regularization;
- evaluate held-out observations;
- inspect calibration and threshold trade-offs.

P-values are not a predictive-quality metric.

# 28. Predictive feature set

We exclude `customer_id` because it is identifier-like.

We include `total_charges` here because prediction can benefit from correlated information even when inferential coefficient interpretation becomes awkward.

This illustrates another key distinction:

> A feature may be undesirable in an explanatory coefficient model but useful in a predictive system.

In [41]:
predictive_features = [
    "gender",
    "senior_citizen",
    "partner",
    "dependents",
    "tenure",
    "phone_service",
    "multiple_lines",
    "internet_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
    "contract",
    "paperless_billing",
    "payment_method",
    "monthly_charges",
    "total_charges",
]

X = df[predictive_features].copy()
y = df["churn_num"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=CFG.test_size,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Train:", X_train.shape)
print("Held-out test:", X_test.shape)
print(f"Train churn rate: {y_train.mean():.4f}")
print(f"Test churn rate:  {y_test.mean():.4f}")

Train: (5634, 19)
Held-out test: (1409, 19)
Train churn rate: 0.2654
Test churn rate:  0.2654


# 29. Leakage-safe preprocessing

Numerical predictors:

- median imputation;
- standardization.

Categorical predictors:

- most-frequent imputation;
- one-hot encoding.

The transformations live **inside** the pipeline.

Therefore, inside cross-validation:

\[
\text{imputer/scaler fit only on training fold}
\]

rather than accidentally learning from validation data.

In [42]:
numeric_features = [
    "senior_citizen",
    "tenure",
    "monthly_charges",
    "total_charges",
]

categorical_features = [
    c for c in predictive_features
    if c not in numeric_features
]

try:
    one_hot = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    )
except TypeError:
    one_hot = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False,
    )

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", one_hot),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

# 30. Candidate predictive models

We include:

1. `DummyClassifier` — baseline;
2. regularized logistic regression;
3. random forest;
4. histogram gradient boosting.

Notice that **logistic regression appears again**, but its role changes.

Inferential GLM:

\[
\hat\beta,\ SE,\ CI,\ OR
\]

Predictive regularized logistic regression:

\[
\arg\min_{\beta}
\left[
-\ell(\beta)
+
\lambda\|\beta\|_2^2
\right]
\]

and evaluate generalization.

In [43]:
models: Dict[str, BaseEstimator] = {
    "Dummy": DummyClassifier(strategy="prior"),

    "Logistic_L2": LogisticRegression(
        C=1.0,
        max_iter=3_000,
        solver="lbfgs",
    ),

    "RandomForest": RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=4,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),

    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=RANDOM_STATE,
    ),
}

pipelines = {
    name: Pipeline([
        ("prep", preprocessor),
        ("model", model),
    ])
    for name, model in models.items()
}

# 31. Repeated cross-validation

A single validation split can be unstable.

Repeated stratified \(K\)-fold CV gives multiple out-of-sample estimates.

Metrics:

- accuracy;
- balanced accuracy;
- F1;
- ROC-AUC;
- average precision;
- log loss.

Different metrics answer different operational questions.

In [44]:
@dataclass
class PredictiveBenchmark:
    models: Dict[str, Pipeline]
    cv: object

    def evaluate(
        self,
        X: pd.DataFrame,
        y: pd.Series,
    ) -> pd.DataFrame:
        scoring = {
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy",
            "f1": "f1",
            "roc_auc": "roc_auc",
            "average_precision": "average_precision",
            "neg_log_loss": "neg_log_loss",
        }

        rows = []

        for name, model in self.models.items():
            print(f"Cross-validating: {name}")

            scores = cross_validate(
                model,
                X,
                y,
                cv=self.cv,
                scoring=scoring,
                n_jobs=-1,
                return_train_score=False,
                error_score="raise",
            )

            row = {"model": name}

            for metric in scoring:
                values = scores[f"test_{metric}"]
                row[f"{metric}_mean"] = values.mean()
                row[f"{metric}_sd"] = values.std(ddof=1)

            rows.append(row)

        return (
            pd.DataFrame(rows)
            .sort_values("roc_auc_mean", ascending=False)
            .reset_index(drop=True)
        )


cv = RepeatedStratifiedKFold(
    n_splits=CFG.cv_splits,
    n_repeats=CFG.cv_repeats,
    random_state=RANDOM_STATE,
)

benchmark = PredictiveBenchmark(
    models=pipelines,
    cv=cv,
)

cv_results = benchmark.evaluate(
    X_train,
    y_train,
)

display(cv_results)

Cross-validating: Dummy
Cross-validating: Logistic_L2
Cross-validating: RandomForest
Cross-validating: HistGradientBoosting


,model,accuracy_mean,accuracy_sd,balanced_accuracy_mean,balanced_accuracy_sd,f1_mean,f1_sd,roc_auc_mean,roc_auc_sd,average_precision_mean,average_precision_sd,neg_log_loss_mean,neg_log_loss_sd
0,Logistic_L2,0.802509,0.010244,0.720096,0.016846,0.593592,0.025249,0.846187,0.012093,0.661047,0.017594,-0.416966,0.013632
1,RandomForest,0.801622,0.008383,0.705178,0.015589,0.571545,0.024579,0.845182,0.011650,0.657262,0.022278,-0.417695,0.012905
2,HistGradientBoosting,0.799669,0.009048,0.711113,0.016354,0.580011,0.024859,0.842096,0.012815,0.657107,0.021065,-0.423528,0.017222
3,Dummy,0.734647,0.000098,0.500000,0.000000,0.000000,0.000000,0.500000,0.000000,0.265353,0.000098,-0.578582,0.000099


### Prediction-focused interpretation

The model-selection question is:

> Which candidate provides the strongest and most stable **out-of-sample** discrimination/probability performance?

This is different from:

\[
H_0:\beta_j=0.
\]

A predictor can have a tiny p-value without materially improving cross-validated ROC-AUC.

# 32. Held-out evaluation

After model comparison on the training set, evaluate fitted models on the untouched held-out set.

We calculate:

- accuracy;
- balanced accuracy;
- precision;
- recall;
- F1;
- ROC-AUC;
- average precision;
- log loss;
- Brier score.

In [45]:
fitted_models = {}
test_probabilities = {}
test_rows = []

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe

    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]

    test_probabilities[name] = prob

    test_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, prob),
        "average_precision": average_precision_score(y_test, prob),
        "log_loss": log_loss(y_test, prob),
        "brier": brier_score_loss(y_test, prob),
    })

test_results = (
    pd.DataFrame(test_rows)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

display(test_results)

,model,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision,log_loss,brier
0,Logistic_L2,0.805536,0.726755,0.657233,0.558824,0.604046,0.841861,0.633429,0.420522,0.138086
1,RandomForest,0.799858,0.705814,0.660839,0.505348,0.572727,0.839789,0.647014,0.421947,0.137238
2,HistGradientBoosting,0.797019,0.711566,0.642857,0.529412,0.580645,0.837871,0.649471,0.426826,0.138564
3,Dummy,0.734564,0.500000,0.000000,0.000000,0.000000,0.500000,0.265436,0.578667,0.194980


# 33. ROC curves

ROC plots:

\[
TPR
\]

against

\[
FPR
\]

as the classification threshold changes.

ROC-AUC measures ranking/discrimination, not probability calibration.

In [46]:
p = figure(
    height=500,
    width=740,
    title="Held-out ROC curves",
    x_axis_label="False positive rate",
    y_axis_label="True positive rate",
    x_range=(0, 1),
    y_range=(0, 1),
)

p.line(
    [0, 1],
    [0, 1],
    line_dash="dashed",
    legend_label="Random ranking",
)

for name, prob in test_probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)

    p.line(
        fpr,
        tpr,
        line_width=2,
        legend_label=f"{name} (AUC={auc:.3f})",
    )

p.legend.location = "bottom_right"
p.legend.click_policy = "hide"
show(p)

# 34. Precision–recall curves

For retention operations, you may care strongly about:

- precision: how many contacted high-risk customers are actual churners?
- recall: how many churners are captured?

PR curves expose this trade-off.

In [47]:
p = figure(
    height=500,
    width=740,
    title="Held-out precision–recall curves",
    x_axis_label="Recall",
    y_axis_label="Precision",
    x_range=(0, 1),
    y_range=(0, 1),
)

for name, prob in test_probabilities.items():
    precision, recall, _ = precision_recall_curve(
        y_test,
        prob,
    )
    ap = average_precision_score(y_test, prob)

    p.line(
        recall,
        precision,
        line_width=2,
        legend_label=f"{name} (AP={ap:.3f})",
    )

p.legend.click_policy = "hide"
show(p)

# 35. Predictive calibration

Suppose a model assigns 100 comparable customers a churn probability near 0.7.

A calibrated model should see approximately 70 of them churn over repeated comparable samples.

Brier score:

\[
BS=
\frac1n
\sum_i
(\hat p_i-y_i)^2.
\]

Lower is better.

In [61]:
p = figure(
    height=500,
    width=680,
    title="Held-out calibration curves",
    x_axis_label="Mean predicted churn probability",
    y_axis_label="Observed churn fraction",
    x_range=(0, 1),
    y_range=(0, 1),
)

p.line(
    [0, 1],
    [0, 1],
    line_dash="dashed",
    legend_label="Perfect calibration",
)

for name, prob in test_probabilities.items():
    frac_pos, mean_pred = calibration_curve(
        y_test,
        prob,
        n_bins=8,
        strategy="quantile",
    )

    p.line(
        mean_pred,
        frac_pos,
        line_width=2,
        legend_label=name,
    )
    p.circle(mean_pred, frac_pos, size=7)

p.legend.click_policy = "hide"
show(p)

# 36. Thresholds and retention decisions

A predictive model returns:

\[
\hat p_i.
\]

The decision threshold should depend on business costs.

Imagine an illustrative rule:

- contacting a customer who would not churn costs **1 unit**;
- missing a churner costs **5 units**.

Then:

\[
Cost(t)
=
FP(t)+5FN(t).
\]

This is only a toy decision model, but it demonstrates that `0.5` is not sacred.

In [56]:
best_non_dummy = (
    test_results.loc[
        test_results["model"] != "Dummy",
        "model",
    ]
    .iloc[0]
)

best_prob = test_probabilities[best_non_dummy]

threshold_rows = []

for threshold in np.arange(0.05, 0.81, 0.025):
    pred = (best_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        pred,
    ).ravel()

    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "fp": fp,
        "fn": fn,
        "illustrative_cost": fp + 5 * fn,
    })

threshold_df = pd.DataFrame(threshold_rows)

best_threshold_row = threshold_df.loc[
    threshold_df["illustrative_cost"].idxmin()
]

display(threshold_df.head())
print("Best model by held-out ROC-AUC:", best_non_dummy)
display(best_threshold_row.to_frame().T)

,threshold,precision,recall,f1,fp,fn,illustrative_cost
0,0.050,0.360000,0.986631,0.527520,656,5,681
1,0.075,0.383475,0.967914,0.549317,582,12,642
2,0.100,0.405498,0.946524,0.567763,519,20,619
3,0.125,0.425245,0.927807,0.583193,469,27,604
4,0.150,0.439024,0.914439,0.593235,437,32,597


Best model by held-out ROC-AUC: Logistic_L2


,threshold,precision,recall,f1,fp,fn,illustrative_cost
5,0.175,0.453441,0.898396,0.602691,405.0,38.0,595.0


In [57]:
p = figure(
    height=420,
    width=720,
    title=f"Illustrative retention cost vs threshold — {best_non_dummy}",
    x_axis_label="Decision threshold",
    y_axis_label="FP + 5×FN",
)

p.line(
    threshold_df["threshold"],
    threshold_df["illustrative_cost"],
    line_width=2,
)
p.circle(
    threshold_df["threshold"],
    threshold_df["illustrative_cost"],
    size=6,
)

p.add_layout(Span(
    location=float(best_threshold_row["threshold"]),
    dimension="height",
    line_dash="dashed",
    line_width=2,
))

show(p)

# 37. Predictive permutation importance

Permutation importance asks:

> How much does predictive score degrade when one feature is randomly shuffled?

This is different from a regression coefficient.

A feature can have:

- high permutation importance;
- no simple linear coefficient interpretation;
- no causal meaning.

We compute importance using held-out ROC-AUC.

In [58]:
best_pipeline = fitted_models[best_non_dummy]

perm = permutation_importance(
    best_pipeline,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=25,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance_df = (
    pd.DataFrame({
        "feature": X_test.columns,
        "importance_mean": perm.importances_mean,
        "importance_sd": perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

display(importance_df)

,feature,importance_mean,importance_sd
0,tenure,0.170572,0.009909
1,internet_service,0.039839,0.005918
2,contract,0.035375,0.005037
3,monthly_charges,0.031907,0.004827
4,total_charges,0.013374,0.003156
5,online_security,0.003763,0.001625
6,streaming_tv,0.003469,0.002353
7,streaming_movies,0.003399,0.001595
8,tech_support,0.003272,0.001701
9,payment_method,0.003132,0.001684


In [60]:
plot_imp = importance_df.sort_values("importance_mean")
source = ColumnDataSource(plot_imp)

p = figure(
    y_range=plot_imp["feature"].tolist(),
    height=520,
    width=780,
    title=f"Predictive permutation importance — {best_non_dummy}",
    x_axis_label="Held-out ROC-AUC degradation after shuffling",
)

p.hbar(
    y="feature",
    right="importance_mean",
    height=0.6,
    source=source,
)

show(p)

# Part III — Put inference and prediction side by side

# 38. Same dataset, different statistical goals

| Dimension | Inferential analytics | Predictive analytics |
|---|---|---|
| Core question | What relationships are supported? | How accurately can unseen customers be scored? |
| Primary object | parameter / estimand | prediction function |
| Example | adjusted contract OR | churn probability |
| Uncertainty | SE, CI, bootstrap | CV variability, held-out error |
| Hypothesis testing | often central | usually secondary |
| p-values | evidence against a null | not a performance metric |
| Validation | assumptions + diagnostics | cross-validation + test set |
| Feature correlation | harms coefficient interpretation | may still be usable |
| Regularization | complicates classical inference | often highly beneficial |
| Causality | requires separate causal assumptions | prediction implies no causality |
| Success criterion | defensible population statement | generalization |

---

## The deepest distinction

### Inferential logistic regression

asks:

\[
\beta_{contract}=?,
\]

\[
CI(\beta_{contract})=?,
\]

\[
OR=e^{\beta_{contract}}=?.
\]

### Predictive logistic regression

asks:

\[
\hat p(x)
\]

and:

> does this probability generalize to unseen customers?

Same mathematical family.

Different objective.

# 39. Why "churn drivers" is a dangerous phrase

Business reports often say:

> The top churn drivers are contract, tenure and monthly charges.

But the phrase **driver** can mean at least three different things.

### A. Marginal association

\[
P(Y\mid X)
\]

differs across levels of \(X\).

### B. Adjusted association

A regression coefficient remains non-zero after adjustment.

### C. Causal effect

Intervening on \(X\) changes \(Y\).

These are not interchangeable.

A predictive feature-importance chart supports neither a clean adjusted coefficient interpretation nor a causal claim.

A careful analyst should say:

- **associated with churn**;
- **predictive of churn**;
- **causal effect of X on churn** only when the study design supports it.

# 40. Suggested inferential report structure

For an assignment or analysis report, use something like:

## Research question

Which measured account and service characteristics are associated with customer churn?

## Data

State:

- source;
- sample size;
- target definition;
- missingness;
- cleaning rules.

## Descriptive analysis

Report:

- churn prevalence + CI;
- group-specific rates + CIs;
- meaningful distributions.

## Unadjusted inference

Report:

- risk differences;
- risk ratios;
- odds ratios;
- chi-square / Welch tests;
- effect sizes;
- bootstrap or permutation evidence.

## Multiple-testing control

When screening many service variables, report BH-adjusted p-values.

## Adjusted inference

Report:

- GLM formula;
- reference categories;
- robust SE;
- adjusted OR + 95% CI;
- standardized probability contrasts.

## Diagnostics

Discuss:

- multicollinearity;
- nonlinear tenure;
- interactions;
- influential observations;
- calibration.

## Prediction

Separately report:

- CV protocol;
- held-out metrics;
- calibration;
- threshold trade-offs.

## Limitations

Explicitly state that the analysis is observational and cannot automatically establish causal effects.

# Part IV — Guided exercises

## Exercise 1 — Online security

Estimate the unadjusted churn:

\[
RD,\ RR,\ OR
\]

for:

- `OnlineSecurity = No`
- versus `OnlineSecurity = Yes`

Restrict to customers with internet service if you want the cleanest service comparison.

Then bootstrap all three measures.

### Question

Does the unadjusted service association remain similar after adjustment in the main GLM?

What could explain attenuation?

In [53]:
# Starter code — Exercise 1

security_subset = df[
    df["online_security"].isin(["No", "Yes"])
].copy()

security_rates = churn_rate_table(
    security_subset,
    "online_security",
)

display(security_rates)

,online_security,count,sum,churn_rate,ci_low,ci_high
0,No,3498,1461,0.417667,0.401423,0.434092
1,Yes,2019,295,0.146112,0.131377,0.162191


## Exercise 2 — Senior citizen status

Run:

\[
SeniorCitizen \times Churn
\]

chi-square inference.

Then fit two logistic models:

### Model A

\[
Churn\sim SeniorCitizen.
\]

### Model B

\[
Churn\sim SeniorCitizen + Contract + Tenure + MonthlyCharges+\cdots.
\]

Compare the ORs.

### Learning objective

Understand the difference between:

- marginal association;
- conditional association.

In [54]:
# Starter code — Exercise 2

senior_test = chi_square_with_cramers_v(
    df,
    "senior_citizen",
)

senior_test

{'feature': 'senior_citizen',
 'chi2': 159.42630036838742,
 'dof': 1,
 'p_value': 1.510066805092378e-36,
 'cramers_v': 0.15045309974200427,
 'min_expected': 303.0523924464007}

## Exercise 3 — Tenure nonlinearity with splines

The quadratic test is only one approach.

Use Patsy's B-spline basis:

```python
bs(tenure_years, df=4)
```

and compare:

\[
\text{linear tenure}
\]

against

\[
\text{spline tenure}.
\]

Plot the resulting adjusted predicted churn curve.

### Question

Does statistical evidence for nonlinearity materially improve prediction?

## Exercise 4 — Bootstrap versus permutation

Using `MonthlyCharges`:

### Bootstrap

Estimate uncertainty in:

\[
\bar X_{churn}
-
\bar X_{nonchurn}.
\]

### Permutation

Shuffle churn labels to test the null of no group association.

Explain why the resulting distributions answer different questions.

## Exercise 5 — Multiple testing

Run chi-square tests on all service fields.

Compare:

- raw \(p<0.05\);
- Bonferroni-adjusted;
- BH-FDR-adjusted.

### Question

Why might BH be preferred during exploratory screening, while a pre-specified confirmatory analysis might use a different error criterion?

## Exercise 6 — Prediction versus inference

Suppose:

- `gender` has a weak adjusted effect;
- removing it barely changes ROC-AUC.

Write two separate conclusions.

### Inferential conclusion

Focus on:

- effect estimate;
- uncertainty;
- adjusted association.

### Predictive conclusion

Focus on:

- incremental generalization benefit.

Do not use predictive irrelevance to prove:

\[
\beta_{gender}=0.
\]

Do not use a significant coefficient to claim predictive necessity.

# Part V — Computational perspective

# 41. Complexity and efficiency

Let:

- \(n\) = customers;
- \(p\) = encoded predictors;
- \(B\) = bootstrap/permutation repetitions;
- \(K\) = CV folds;
- \(R\) = repeats;
- \(T\) = trees.

## Resampling

A simple statistic bootstrap/permutation loop costs:

\[
\Theta(Bn).
\]

## Model-refit bootstrap

Approximately:

$$\Theta(B \times C_{\text{fit}}(n,p))$$



This can become expensive quickly.

## Cross-validation

Approximately:

\[
K\times R
\]

times the per-model fit cost.

## Dense one-hot encoding

Memory can approach:

\[
\Theta(np).
\]

For this 7k-row dataset that is manageable.

For very high-cardinality production data, prefer sparse representations where compatible.

---

## Efficiency tips

During notebook development:

```python
bootstrap_reps = 500
permutation_reps = 1000
cv_repeats = 1
n_estimators = 150
```

For final analysis, increase the settings after validating the workflow.

Also:

- vectorize simple statistics;
- parallelize independent CV/model jobs;
- avoid refitting unchanged preprocessors unnecessarily;
- keep inferential and predictive feature sets intentionally separate.

# Part VI — Advanced extensions beyond the core notebook

# 42. Where to go next

For a more advanced inferential project, consider:

### A. Multiple imputation

Instead of complete-case analysis:

\[
M
\]

imputed datasets + Rubin's rules.

### B. Restricted cubic splines

Model tenure and monthly charges flexibly.

### C. Penalized logistic regression

Useful when the feature space is large, but classical p-values require care.

### D. Cluster-robust standard errors

If multiple customers belong to the same household/account/region.

### E. Time-to-event modelling

Churn is naturally a **survival-analysis** problem if event time and censoring are available.

A richer dataset would allow:

\[
h(t\mid X)
\]

using Cox proportional hazards or parametric survival models.

### F. Causal inference

If the question changes from:

> Who churns?

to:

> What intervention reduces churn?

you need treatment/intervention data and methods such as:

- randomized experiments;
- propensity-score approaches;
- inverse-probability weighting;
- doubly robust estimation;
- causal forests / heterogeneous treatment effects.

The current Kaggle table alone is not enough to identify many such causal effects credibly.

# 43. Final conceptual summary

## Inferential analytics

The notebook's core workflow was:

```text
Population / estimand
        ↓
Descriptive estimate
        ↓
Effect magnitude
        ↓
Confidence interval
        ↓
Hypothesis / permutation evidence
        ↓
Multiple-testing control
        ↓
Adjusted regression
        ↓
Odds ratios
        ↓
Standardized probabilities
        ↓
Interaction / nonlinearity
        ↓
Diagnostics
        ↓
Limitations
```

The objective is:

$\boxed{\text{Inline boxed text}}$


---

## Predictive analytics

The secondary workflow was:

```text
Target
  ↓
Train/test separation
  ↓
Leakage-safe preprocessing
  ↓
Candidate algorithms
  ↓
Repeated CV
  ↓
Held-out evaluation
  ↓
Calibration
  ↓
Threshold / decision policy
```

The objective is:

\[
\boxed{
\text{Generalize accurately to unseen customers}
}
\]

---

# The DSA3361 lesson

Do not begin with:

> Which ML model should I use?

Begin with:

> **What statistical question am I trying to answer?**

If the question is inferential, you need:

- a target population;
- an estimand;
- sampling assumptions;
- uncertainty quantification;
- adjustment logic;
- diagnostics;
- cautious interpretation.

If the question is predictive, you need:

- proper validation;
- leakage control;
- out-of-sample metrics;
- calibration;
- operational decision rules.

Strong data science knows when these goals overlap—and when they absolutely do not.

# 44. Dataset/source notes

Primary dataset:

- Kaggle — **Telco Customer Churn**
- Dataset handle: `blastchar/telco-customer-churn`
- File: `WA_Fn-UseC_-Telco-Customer-Churn.csv`

Kaggle describes the data as customer-level telecom records containing churn status, subscribed services, account information, contract/payment details and demographics.

The notebook is intended for educational statistical analysis. Dataset licensing/usage terms remain those of the original dataset authors/Kaggle listing.